# 🛡️ Raspberry Pi Edge Intrusion Prevention System (IPS) - CICIoT2023
### Tabular Foundation Models, Edge Optimization & Real-Time Mitigation
Based on the IEEE Research Paper: *"Lightweight ML-based Intrusion Detection Systems for IoT: A Comprehensive Analysis"* (IIIT Allahabad, 2025).

---
### Key Objectives & System Architecture:
1. **Edge Hardware Constraint**: Tailored for resource-constrained IoT gateways (e.g., **Raspberry Pi 3B+ / 4 / 5**, ARM Cortex-A53 quad-core, 1GB RAM).
2. **Two-Phase Architecture**:
   - **Phase 1**: Circular ring-buffer memory management (preventing RAM exhaustion) & feature extraction prioritizing **Inter-Arrival Time (IAT)**.
   - **Phase 2**: Sub-millisecond threat classification (<1ms latency budget).
3. **Active Prevention vs Passive Detection**:
   - Unlike an IDS which only logs attacks, this **IPS** actively triggers automated Linux firewall mitigations (`iptables` / `nftables` DROP/REJECT rules, port quarantines, ARP cache freeze).
4. **Tabular Foundation & Ensemble Models**:
   - Evaluates **FT-Transformer (Tabular Foundation Model)**, **Stacking Ensemble**, **Voting Ensemble**, **Decision Tree (Edge Champion)**, and **Edge XGBoost / LightGBM**.



## Step 1: Environment Setup & Dependencies



In [ ]:
import os
import sys
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Verify imports
import torch
import xgboost as xgb
import lightgbm as lgb
from src.data_loader import prepare_or_load_data, transform_targets
from src.config import FEATURE_COLUMNS, LABEL_COLUMN

print(f"PyTorch: {torch.__version__} | XGBoost: {xgb.__version__} | LightGBM: {lgb.__version__}")



## Step 2: Load Stratified Parquet Dataset
We load our balanced, stratified samples (69,611 training records, 16,598 test records) preserving rare attack categories.



In [ ]:
train_df, val_df, test_df = prepare_or_load_data()
print(f"Train Shape: {train_df.shape} | Test Shape: {test_df.shape}")



## Step 3: Raspberry Pi Edge Benchmark (Accuracy vs Latency vs Size)
We benchmark the Tabular Foundation Transformer, Stacking Ensemble, Voting Ensemble, and Edge Decision Tree.



In [ ]:
# Run the benchmark engine
!python benchmark_rpi_edge.py



## Step 4: Edge Visual Results & Pareto Frontier Gallery



In [ ]:
print("=== 1. Pareto Frontier: Accuracy vs Inference Latency ===")
display(Image(filename="results/figures/eval_rpi_latency_vs_accuracy.png", width=750))

print("\n=== 2. Edge Hardware Resource Footprint (Model Size & Latency) ===")
display(Image(filename="results/figures/eval_rpi_resource_footprint.png", width=850))

print("\n=== 3. Key Feature Importance Ranking (Highlighting IAT) ===")
display(Image(filename="results/figures/eval_rpi_feature_importance_iat.png", width=750))



## Step 5: Live Demonstration of Autonomous Raspberry Pi IPS
Simulate incoming network packet flows and watch the IPS classify threats and generate active Linux firewall rules in real-time.



In [ ]:
from src.raspberry_pi_ips import RaspberryPiIntrusionPreventionSystem

ips = RaspberryPiIntrusionPreventionSystem(model_task="category")

# Simulate 5 incoming test flows
mock_flows = test_df.sample(n=5, random_state=101)
test_ips = ["192.168.1.110", "10.0.0.52", "172.16.10.4", "45.33.32.1", "192.168.1.1"]
test_ports = [80, 22, 443, 23, 8080]

print("="*85)
print("              RASPBERRY PI AUTONOMOUS IPS ACTIVE PREVENTIONS")
print("="*85)
for idx, (_, row) in enumerate(mock_flows.iterrows()):
    src_ip = test_ips[idx]
    dport = test_ports[idx]
    verdict = ips.inspect_and_mitigate(row.to_dict(), source_ip=src_ip, dest_port=dport)
    print(f"[{idx+1}] Inbound {src_ip}:{dport} -> Verdict: {verdict['verdict']} ({verdict['confidence']}) | Action: {verdict['ips_action']}")
    print(f"    Firewall Rule: {verdict['firewall_rule']}")
    print(f"    Mitigation:    {verdict['mitigation_summary']}\n")

ips.print_telemetry()

